In [ ]:
import bz2
import re
import os
import pandas as pd
from collections import defaultdict

PATH = r"C:\Users\jaden\Downloads\latest-all.nt.bz2"
OUT_CSV = r"C:\Users\jaden\Downloads\wikidata_temporal_rows_v2.csv"

# How often to checkpoint work
PRINT_EVERY = 1_000_000
FLUSH_EVERY = 10_000_000

ENTITY_TO_STMT_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/(Q\d+)> '
    r'<http://www\.wikidata\.org/prop/(P\d+)> '
    r'<http://www\.wikidata\.org/entity/statement/([^>]+)> \.$'
)

STMT_TO_OBJ_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/statement/([^>]+)> '
    r'<http://www\.wikidata\.org/prop/statement/(P\d+)> '
    r'<http://www\.wikidata\.org/entity/(Q\d+)> \.$'
)

STMT_TO_TIME_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/statement/([^>]+)> '
    r'<http://www\.wikidata\.org/prop/qualifier/(P580|P582)> '
    r'"([^"]+)"\^\^<http://www\.w3\.org/2001/XMLSchema#dateTime> \.$'
)

STMT_DEPRECATED_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/statement/([^>]+)> '
    r'<http://wikiba\.se/ontology#rank> '
    r'<http://wikiba\.se/ontology#DeprecatedRank> \.$'
)


def statement_is_writable(data):
    """
    A statement is ready to write if:
    - not deprecated
    - no predicate mismatch
    - has subject, pred, obj
    - has at least one of start/end
    """
    if data.get("deprecated", False):
        return False
    if data.get("pred_mismatch", False):
        return False
    if "subject" not in data or "pred" not in data or "obj" not in data:
        return False
    if "start" not in data and "end" not in data:
        return False
    return True


def flush_ready_rows(statements, out_csv, write_header):
    """
    Pull completed rows out of the in-memory dict, append them to CSV,
    and delete them from memory.

    Returns:
        num_rows_written, new_write_header
    """
    rows = []
    stmt_ids_to_delete = []

    for stmt_id, data in statements.items():
        if statement_is_writable(data):
            rows.append({
                "subject": data["subject"],
                "pred": data["pred"],
                "obj": data["obj"],
                "start": data.get("start"),
                "end": data.get("end"),
            })
            stmt_ids_to_delete.append(stmt_id)

        # Also safe to drop permanently bad rows to save memory
        elif data.get("deprecated", False) or data.get("pred_mismatch", False):
            stmt_ids_to_delete.append(stmt_id)

    if rows:
        df = pd.DataFrame(rows, columns=["subject", "pred", "obj", "start", "end"])
        df.to_csv(
            out_csv,
            mode="a",
            index=False,
            header=write_header
        )
        write_header = False

    for stmt_id in stmt_ids_to_delete:
        del statements[stmt_id]

    return len(rows), write_header



# If output already exists, remove it so this run starts clean
if os.path.exists(OUT_CSV):
    os.remove(OUT_CSV)

statements = defaultdict(dict)
line_count = 0
total_rows_written = 0
write_header = True

try:
    with bz2.open(PATH, "rt", encoding="utf-8") as f:
        for raw_line in f:
            line_count += 1

            if line_count % PRINT_EVERY == 0:
                print(
                    f"Processed {line_count:,} lines | "
                    f"in-memory statements: {len(statements):,} | "
                    f"rows written: {total_rows_written:,}"
                )

            line = raw_line.strip()

            m = ENTITY_TO_STMT_RE.match(line)
            if m:
                subj, pred, stmt_id = m.groups()
                statements[stmt_id]["subject"] = subj
                statements[stmt_id]["pred"] = pred

            else:
                m = STMT_TO_OBJ_RE.match(line)
                if m:
                    stmt_id, pred2, obj = m.groups()
                    statements[stmt_id]["obj"] = obj
                    if "pred" in statements[stmt_id] and statements[stmt_id]["pred"] != pred2:
                        statements[stmt_id]["pred_mismatch"] = True
                    else:
                        statements[stmt_id]["pred"] = pred2

                else:
                    m = STMT_TO_TIME_RE.match(line)
                    if m:
                        stmt_id, time_prop, dt = m.groups()
                        if time_prop == "P580":
                            statements[stmt_id]["start"] = dt
                        else:
                            statements[stmt_id]["end"] = dt

                    else:
                        m = STMT_DEPRECATED_RE.match(line)
                        if m:
                            stmt_id = m.group(1)
                            statements[stmt_id]["deprecated"] = True

            # Periodically flush ready statements to disk
            if line_count % FLUSH_EVERY == 0:
                rows_written, write_header = flush_ready_rows(
                    statements, OUT_CSV, write_header
                )
                total_rows_written += rows_written

                print(
                    f"Checkpoint flush at {line_count:,} lines | "
                    f"wrote {rows_written:,} rows | "
                    f"remaining in memory: {len(statements):,}"
                )

except EOFError:
    print("Warning: truncated bz2 file reached. Using the valid portion that was read.")

# Final flush after loop ends
rows_written, write_header = flush_ready_rows(statements, OUT_CSV, write_header)
total_rows_written += rows_written

print(f"Finished.")
print(f"Lines processed: {line_count:,}")
print(f"Total rows written: {total_rows_written:,}")
print(f"Remaining in-memory statements after final flush: {len(statements):,}")
print(f"Output CSV: {OUT_CSV}")

Processed 1,000,000 lines | in-memory statements: 60,040 | rows written: 0
Processed 2,000,000 lines | in-memory statements: 115,379 | rows written: 0
Processed 3,000,000 lines | in-memory statements: 180,799 | rows written: 0
Processed 4,000,000 lines | in-memory statements: 238,907 | rows written: 0
Processed 5,000,000 lines | in-memory statements: 294,858 | rows written: 0
Processed 6,000,000 lines | in-memory statements: 357,200 | rows written: 0
Processed 7,000,000 lines | in-memory statements: 425,491 | rows written: 0
Processed 8,000,000 lines | in-memory statements: 502,979 | rows written: 0
Processed 9,000,000 lines | in-memory statements: 582,338 | rows written: 0
Processed 10,000,000 lines | in-memory statements: 659,299 | rows written: 0
Checkpoint flush at 10,000,000 lines | wrote 16,719 rows | remaining in memory: 637,402
Processed 11,000,000 lines | in-memory statements: 715,000 | rows written: 16,719
Processed 12,000,000 lines | in-memory statements: 791,069 | rows writ

In [ ]:
# read from csv
df = pd.read_csv(OUT_CSV, dtype=str)

In [ ]:
import pandas as pd

# If not already parsed, convert to datetime
df["start_dt"] = pd.to_datetime(df["start"], errors="coerce", utc=True)
df["end_dt"] = pd.to_datetime(df["end"], errors="coerce", utc=True)

has_start = df["start_dt"].notna()
has_end = df["end_dt"].notna()

only_start = (has_start & ~has_end).sum()
only_end = (~has_start & has_end).sum()
both = (has_start & has_end).sum()
neither = (~has_start & ~has_end).sum()

total = len(df)

print(f"Total rows: {total:,}")
print(f"Only start: {only_start:,}")
print(f"Only end:   {only_end:,}")
print(f"Both:       {both:,}")
print(f"Neither:    {neither:,}")

['Q4916', 'Q232415', 'Q212', 'Q230', 'Q423', 'Q1079522', 'Q12971', 'Q12967', 'Q55008046', 'Q12973', 'Q445553', 'Q12973', 'Q12976', 'Q12976', 'Q3911', 'Q155004', 'Q18434995', 'Q950958', 'Q476596', 'Q336599']
1.0
Total rows: 17,247
Only start: 6,830
Only end:   1,718
Both:       8,000
Neither:    699


In [ ]:
# drop 
df = df[df["start"].notna() & df["end"].notna()].copy()
df.head()

,subject,pred,obj,start,end,start_dt,end_dt
1,Q31,P38,Q232415,1830-01-01T00:00:00Z,2002-01-01T00:00:00Z,1830-01-01 00:00:00+00:00,2002-01-01 00:00:00+00:00
5,Q31,P35,Q1079522,1831-02-25T00:00:00Z,1831-07-20T00:00:00Z,1831-02-25 00:00:00+00:00,1831-07-20 00:00:00+00:00
6,Q31,P35,Q12971,1831-06-04T00:00:00Z,1865-12-10T00:00:00Z,1831-06-04 00:00:00+00:00,1865-12-10 00:00:00+00:00
7,Q31,P35,Q12967,1865-12-17T00:00:00Z,1909-12-17T00:00:00Z,1865-12-17 00:00:00+00:00,1909-12-17 00:00:00+00:00
8,Q31,P35,Q55008046,1909-12-23T00:00:00Z,1934-02-17T00:00:00Z,1909-12-23 00:00:00+00:00,1934-02-17 00:00:00+00:00


In [ ]:
import bz2
import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd


DUMP_PATH = r"C:\Users\jaden\Downloads\latest-all.nt.bz2"
OUTPUT_JSON = r"C:\Users\jaden\Downloads\temporal_year_db.json"


LABEL_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/(Q\d+|P\d+)> '
    r'<http://www\.w3\.org/2000/01/rdf-schema#label> '
    r'"((?:[^"\\]|\\.)*)"@en \.$'
)

DESC_RE = re.compile(
    r'^<http://www\.wikidata\.org/entity/(Q\d+|P\d+)> '
    r'<http://schema\.org/description> '
    r'"((?:[^"\\]|\\.)*)"@en \.$'
)


def normalize_id(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith("http://www.wikidata.org/entity/"):
        s = s.rsplit("/", 1)[-1]
    return s


def year_from_value(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None

    i = 0
    if s[0] in "+-":
        i = 1

    digits = []
    while i < len(s) and s[i].isdigit():
        digits.append(s[i])
        i += 1

    if not digits:
        return None

    year = int("".join(digits))
    return -year if s[0] == "-" else year


def load_json_or_empty(path):
    """Load a JSON file if it exists, else return an empty dict."""
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_json_atomic(data, path):
    """Write JSON safely by using a temp file then replacing."""
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp_path, path)


def decode_nt_string(s):
    """Decode escaped unicode sequences from N-Triples strings."""
    return bytes(s, "utf-8").decode("unicode_escape")


def q_entry_complete(x):
    """A Q cache entry is complete if it has both label and description."""
    return (
        isinstance(x, dict)
        and x.get("label") is not None
        and x.get("description") is not None
    )


def build_lookup_from_dump(
    dump_path,
    ids_needed,
    ent_cache_path,
    rel_cache_path,
    scan_dump=True,
    save_every=500_000,
):
    """
    Build a lookup for mixed Q and P ids.

    Parameters
    ----------
    dump_path : str
        Path to the bz2 Wikidata dump.
    ids_needed : iterable
        IDs like Q42, P31, etc.
    ent_cache_path : str
        JSON cache path for Q ids.
        Format:
            {
                "Q42": {"label": "...", "description": "..."}
            }
    rel_cache_path : str
        JSON cache path for P ids.
        Format:
            {
                "P31": "instance of"
            }
    scan_dump : bool
        If False:
            return only what is currently in the caches.
        If True:
            read caches first, then scan the dump for missing data and
            update caches as matches are found.
    save_every : int
        How many lines between periodic cache saves while scanning.

    Returns
    -------
    lookup : dict
        Mixed dictionary:
        - Q ids -> {"label": ..., "description": ...}
        - P ids -> "label"
    """

    # Normalize all requested ids
    normalized_ids = set()
    for x in ids_needed:
        nx = normalize_id(x)
        if nx is not None:
            normalized_ids.add(nx)

    # Split into entities and relations
    q_ids = {x for x in normalized_ids if x.startswith("Q")}
    p_ids = {x for x in normalized_ids if x.startswith("P")}

    # Load caches
    ent_cache = load_json_or_empty(ent_cache_path)
    rel_cache = load_json_or_empty(rel_cache_path)

    # Build initial lookup from cache contents
    lookup = {}

    missing_q = set()
    for qid in q_ids:
        cached = ent_cache.get(qid)
        if isinstance(cached, dict):
            lookup[qid] = {
                "label": cached.get("label"),
                "description": cached.get("description"),
            }
        else:
            lookup[qid] = {"label": None, "description": None}

        if not q_entry_complete(lookup[qid]):
            missing_q.add(qid)

    missing_p = set()
    for pid in p_ids:
        cached = rel_cache.get(pid)
        if isinstance(cached, str) and cached:
            lookup[pid] = cached
        else:
            lookup[pid] = None
            missing_p.add(pid)

    # If caller only wants cached contents, stop here
    if not scan_dump:
        return lookup

    # If cache already had everything, no need to scan
    if not missing_q and not missing_p:
        return lookup

    lines_processed = 0
    last_save_line = 0

    try:
        with bz2.open(dump_path, "rt", encoding="utf-8") as f:
            for line in f:
                lines_processed += 1

                # Cheap filter before regex
                if "wikidata.org/entity/" not in line:
                    continue

                # First try label lines
                m = LABEL_RE.match(line)
                if m:
                    eid, raw_label = m.groups()
                    label = decode_nt_string(raw_label)

                    # Property label
                    if eid in missing_p:
                        rel_cache[eid] = label
                        lookup[eid] = label
                        missing_p.remove(eid)

                    # Entity label
                    elif eid in missing_q:
                        if eid not in ent_cache or not isinstance(ent_cache[eid], dict):
                            ent_cache[eid] = {"label": None, "description": None}

                        ent_cache[eid]["label"] = label
                        lookup[eid]["label"] = label

                        if q_entry_complete(ent_cache[eid]):
                            missing_q.remove(eid)

                else:
                    # Then try description lines for Q ids
                    m = DESC_RE.match(line)
                    if m:
                        eid, raw_desc = m.groups()

                        if eid in missing_q:
                            desc = decode_nt_string(raw_desc)

                            if eid not in ent_cache or not isinstance(ent_cache[eid], dict):
                                ent_cache[eid] = {"label": None, "description": None}

                            ent_cache[eid]["description"] = desc
                            lookup[eid]["description"] = desc

                            if q_entry_complete(ent_cache[eid]):
                                missing_q.remove(eid)

                # Stop early once everything is found
                if not missing_q and not missing_p:
                    break

                # Periodically save caches during long scans
                if lines_processed - last_save_line >= save_every:
                    save_json_atomic(ent_cache, ent_cache_path)
                    save_json_atomic(rel_cache, rel_cache_path)
                    last_save_line = lines_processed
                    print(
                        f"Processed {lines_processed:,} lines | "
                        f"remaining Q: {len(missing_q):,} | "
                        f"remaining P: {len(missing_p):,}"
                    )

    except EOFError:
        print("Warning: dump appears truncated; returning partial lookup.")

    # Final save after scan
    save_json_atomic(ent_cache, ent_cache_path)
    save_json_atomic(rel_cache, rel_cache_path)

    return lookup


def build_year_json(df, dump_path, output_json_path):
    required = {"subject", "pred", "obj", "start", "end"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = df.copy()
    df["subject"] = df["subject"].map(normalize_id)
    df["pred"] = df["pred"].map(normalize_id)
    df["obj"] = df["obj"].map(normalize_id)

    df = df[df["start"].notna() & df["end"].notna()].copy()

    ids_needed = set(df["subject"]) | set(df["pred"]) | set(df["obj"])
    print(f"Need to resolve {len(ids_needed):,} unique subject/predicate IDs from dump...")

    ent_cache_path = "wikidata_entity_label_cache.json"
    rel_cache_path = "wikidata_relation_label_cache.json"

    lookup = build_lookup_from_dump(dump_path, ids_needed, ent_cache_path, rel_cache_path, scan_dump=False, save_every=500_000)

    def subject_info(qid):
        info = lookup.get(qid)
        if not info:
            return None
        label = info.get("label")
        description = info.get("description")
        if not label or not description:
            return None
        return {"label": label, "description": description}

    def relation_text(pid):
        info = lookup.get(pid)
        if not info:
            return None
        return info.get("label")


    df["head"] = df["subject"].map(subject_info)
    df["relation"] = df["pred"].map(relation_text)
    df["tail"] = df["obj"].map(subject_info)
    df["start_year"] = df["start"].map(year_from_value)
    df["end_year"] = df["end"].map(year_from_value)

    print("head:", df["head"].notna().sum())
    print("relation:", df["relation"].notna().sum())
    print("tail:", df["tail"].notna().sum())
    print("start_year:", df["start_year"].notna().sum())
    print("end_year:", df["end_year"].notna().sum())
    print("total rows:", len(df))

    print("head & relation:", (df["head"].notna() & df["relation"].notna()).sum())
    print("years ok:", (df["start_year"].notna() & df["end_year"].notna()).sum())

    df = df[
        df["head"].notna()
        & df["relation"].notna()
        & df["tail"].notna()
        & df["start_year"].notna()
        & df["end_year"].notna()
    ].copy()

    print(f"Rows with everything: {len(df)}")


    df["start_year"] = df["start_year"].astype(int)
    df["end_year"] = df["end_year"].astype(int)
    df = df[df["start_year"] <= df["end_year"]].copy()

    year_db = defaultdict(list)
    month_db = defaultdict(list)


    def parse_year_month(x):
        """
        Handles strings like:
        1843-##-##
        1843-05-##
        1843-05-01T00:00:00Z
        +1843-05-01T00:00:00Z

        Returns:
        (year, month) where month can be None if unknown
        """
        s = str(x).strip()

        if s[0] == "+":
            s = s[1:]

        year = int(s[:4])

        month = None
        if len(s) >= 7:
            mm = s[5:7]
            if mm.isdigit() and mm != "00":
                month = int(mm)

        return year, month


    for _, row in df.iterrows():
        triple = {
            "head": row["head"],
            "relation": row["relation"],
            "tail": row["tail"],
            "start": row["start"],
            "end": row["end"],
        }

        start_year, start_month = parse_year_month(row["start"])
        end_year, end_month = parse_year_month(row["end"])

        # Fill missing months for interval expansion
        start_month = 1 if start_month is None else start_month
        end_month = 12 if end_month is None else end_month

        # Year DB
        for year in range(start_year, end_year + 1):
            year_db[str(year)].append(triple)

        # Month DB
        y, m = start_year, start_month
        while (y < end_year) or (y == end_year and m <= end_month):
            month_db[f"{y:04d}-{m:02d}"].append(triple)

            m += 1
            if m == 13:
                m = 1
                y += 1

    year_db = dict(year_db)
    month_db = dict(month_db)

    output_json_path = Path(output_json_path)
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(year_db, f, indent=2, ensure_ascii=False)
    
    output_json_path_monthly = output_json_path.with_name(
        output_json_path.stem + "_monthly.json"
    )
    with open(output_json_path_monthly, "w", encoding="utf-8") as f:
        json.dump(month_db, f, indent=2, ensure_ascii=False)

    print(f"Saved JSON DB to: {output_json_path}. _monthly for monthly one")
    print(f"Rows kept: {len(df):,}")
    print(f"Years written: {len(year_db):,}")
    print(f"Months written: {len(month_db):,}")
    print(f"Total triples: {sum(len(v) for v in year_db.values()):,}")
    

    return year_db, df, lookup


year_db, filtered_df, lookup = build_year_json(
    df=df,
    dump_path=DUMP_PATH,
    output_json_path=OUTPUT_JSON,
)

Need to resolve 7,511 unique subject/predicate IDs from dump...
Processed 10,000,000 lines
head: 8467
relation: 0
tail: 187
start_year: 8564
end_year: 8564
total rows: 8564
head & relation: 0
years ok: 8564
Saved JSON DB to: C:\Users\jaden\Downloads\temporal_year_db.json
Rows kept: 0
Years written: 0
Total year-indexed triples: 0


In [ ]:
import json
from pathlib import Path

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt


# =========================
# User config
# =========================
DATA_PATH = r"C:\Users\jaden\Downloads\temporal_year_db.json"
YEAR = 1793
MAX_EDGES_TO_DRAW = 300
ONLY_LARGEST_COMPONENT = False


# =========================
# Helpers
# =========================
def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def maybe_reduce_for_plot(
    G: nx.MultiDiGraph,
    max_edges: int | None = None,
    largest_component_only: bool = False,
) -> nx.MultiDiGraph:
    H = G

    if largest_component_only and G.number_of_nodes() > 0:
        components = list(nx.connected_components(G.to_undirected()))
        if components:
            largest = max(components, key=len)
            H = G.subgraph(largest).copy()

    if max_edges is not None and H.number_of_edges() > max_edges:
        trimmed = nx.MultiDiGraph()
        for i, (u, v, k, data) in enumerate(H.edges(keys=True, data=True)):
            if i >= max_edges:
                break
            trimmed.add_node(u)
            trimmed.add_node(v)
            trimmed.add_edge(u, v, key=k, **data)
        H = trimmed

    return H


def build_dataframe(year_triples: list[dict]) -> pd.DataFrame:
    rows = []
    for t in year_triples:
        head = t.get("head", {})
        rows.append(
            {
                "subject_text": head.get("label"),
                "pred_text": t.get("relation"),
                "obj_text": t.get("tail"),
                "start": t.get("start"),
                "end": t.get("end"),
            }
        )

    df = pd.DataFrame(rows)
    df = df[
        df["subject_text"].notna()
        & df["pred_text"].notna()
        & df["obj_text"].notna()
    ].reset_index(drop=True)
    return df


def build_graph(df: pd.DataFrame) -> nx.MultiDiGraph:
    G = nx.MultiDiGraph()

    for _, row in df.iterrows():
        G.add_edge(
            row["subject_text"],
            row["obj_text"],
            label=row["pred_text"],
            start=row["start"],
            end=row["end"],
        )

    return G


def draw_graph(G: nx.MultiDiGraph, year: int) -> None:
    if G.number_of_nodes() == 0:
        print(f"No triples to display for {year}.")
        return

    plt.figure(figsize=(16, 12))
    pos = nx.spring_layout(G, seed=42, k=1.2)

    nx.draw_networkx_nodes(G, pos, node_size=900)
    nx.draw_networkx_labels(G, pos, font_size=9)
    nx.draw_networkx_edges(
        G,
        pos,
        arrows=True,
        arrowstyle="->",
        arrowsize=15,
        connectionstyle="arc3,rad=0.08",
    )

    edge_labels = {
        (u, v, k): data["label"]
        for u, v, k, data in G.edges(keys=True, data=True)
    }

    nx.draw_networkx_edge_labels(
        G,
        pos,
        edge_labels=edge_labels,
        font_size=8,
        rotate=False,
        label_pos=0.5,
    )

    plt.title(f"Wikidata temporal graph for {year}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# =========================
# Main
# =========================
def main():
    year_db = load_json(DATA_PATH)
    year_triples = year_db.get(str(YEAR), [])

    print(f"Triples for {YEAR}: {len(year_triples):,}")
    if not year_triples:
        return

    df = build_dataframe(year_triples)
    print("\nPreview:")
    print(df.head(20).to_string(index=False))

    G_full = build_graph(df)
    G_draw = maybe_reduce_for_plot(
        G_full,
        max_edges=MAX_EDGES_TO_DRAW,
        largest_component_only=ONLY_LARGEST_COMPONENT,
    )

    print(f"\nGraph nodes: {G_full.number_of_nodes():,}")
    print(f"Graph edges: {G_full.number_of_edges():,}")

    if G_draw.number_of_edges() != G_full.number_of_edges():
        print(
            f"Drawing reduced graph with {G_draw.number_of_nodes():,} nodes and "
            f"{G_draw.number_of_edges():,} edges."
        )

    draw_graph(G_draw, YEAR)
    return df, G_full


if __name__ == "__main__":
    df, G = main()

Triples for 1793: 0


TypeError: cannot unpack non-iterable NoneType object